# 第9章：RAG 基础

> 🎯 构建检索增强生成系统

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

load_dotenv()
llm = ChatOpenAI(
    base_url=os.getenv('LM_STUDIO_BASE_URL', 'http://localhost:1234/v1'),
    api_key='lm-studio',
    model=os.getenv('LM_STUDIO_MODEL', 'qwen2.5-7b-instruct')
)
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

## 1. 准备知识库

In [ ]:
# 模拟公司知识库
docs = [
    Document(page_content='公司成立于2020年，总部在北京'),
    Document(page_content='员工福利包括：五险一金、年假15天、免费午餐'),
    Document(page_content='请假流程：提前一天在OA系统提交申请'),
    Document(page_content='报销标准：出差住宿每晚不超过500元'),
]

vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})
print('知识库准备完成')

## 2. 构建RAG Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

template = '''基于以下信息回答问题。如果信息中没有答案，说"抱歉，我没有找到相关信息"。

信息：
{context}

问题：{question}
'''

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return '\n'.join(doc.page_content for doc in docs)

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 3. 测试RAG

In [ ]:
questions = [
    '公司有哪些员工福利？',
    '怎么请假？',
    '出差住宿报销标准是多少？',
    '公司有多少员工？'  # 知识库中没有的问题
]

for q in questions:
    print(f'问：{q}')
    print(f'答：{rag_chain.invoke(q)}')
    print()

## 4. 查看检索结果

In [ ]:
# 查看检索到的文档
question = '员工福利有哪些？'
retrieved_docs = retriever.invoke(question)

print(f'问题: {question}')
print('\n检索到的文档:')
for i, doc in enumerate(retrieved_docs, 1):
    print(f'{i}. {doc.page_content}')

## 📝 小结
RAG流程：
1. **索引**：文档 → 向量 → 存储
2. **检索**：问题 → 向量 → 相似搜索
3. **生成**：问题 + 检索结果 → LLM → 回答